# Privacy & PII Governance

**Use case:** Detect and redact sensitive information before AI processing.

This notebook demonstrates the Responsible AI / Governance concept in a simple end-to-end flow.

**Total steps:** 10

## Installation

```bash
pip install pandas numpy scikit-learn matplotlib langchain-openai python-dotenv
```

## Step 1 - Create sample customer text

In [ ]:
records = pd.DataFrame({'text':['My email is ravi@example.com and phone is 9876543210','Order 1002 is delayed','Contact me at meera@test.com','My card number is 4111 1111 1111 1111']})
records


## Step 2 - Define simple PII patterns

In [ ]:
import re
email_pattern = re.compile(r'[\w\.-]+@[\w\.-]+')
phone_pattern = re.compile(r'\b\d{10}\b')
card_pattern = re.compile(r'\b(?:\d[ -]*?){13,16}\b')


## Step 3 - Detect email addresses

In [ ]:
records['has_email'] = records['text'].apply(lambda x:bool(email_pattern.search(x)))


## Step 4 - Detect phone numbers

In [ ]:
records['has_phone'] = records['text'].apply(lambda x:bool(phone_pattern.search(x)))


## Step 5 - Detect possible payment-card data

In [ ]:
records['has_card'] = records['text'].apply(lambda x:bool(card_pattern.search(x)))


## Step 6 - Create a PII risk flag

In [ ]:
records['pii_detected'] = records[['has_email','has_phone','has_card']].any(axis=1)
print(records)


## Step 7 - Redact email

In [ ]:
records['redacted'] = records['text'].apply(lambda x:email_pattern.sub('<EMAIL>',x))


## Step 8 - Redact phone and card

In [ ]:
records['redacted'] = records['redacted'].apply(lambda x:phone_pattern.sub('<PHONE>',x))
records['redacted'] = records['redacted'].apply(lambda x:card_pattern.sub('<PAYMENT_CARD>',x))
print(records[['text','redacted']])


## Step 9 - Define privacy decision

In [ ]:
records['privacy_action'] = np.where(records['pii_detected'],'REDACT_AND_CONTINUE','ALLOW')
print(records[['pii_detected','privacy_action']])


## Step 10 - Save privacy evidence

In [ ]:
records.to_csv('privacy_redaction_evidence.csv',index=False)
print('Saved privacy_redaction_evidence.csv')
